In [1]:
pip install streamlit fpdf2

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.0/81.0 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 74.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 327.1/327.1 kB 26.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 101.9 MB/s eta 0:00:00


In [3]:
import streamlit as st
from fpdf import FPDF
import smtplib
from email.mime.multipart import MIMEMultipart
from email.mime.text import MIMEText
from email.mime.base import MIMEBase
from email import encoders
import os
from datetime import datetime

In [4]:
# ==========================================
# CONFIGURATION & CONSTANTS
# ==========================================
# IMPORTANT: Replace these with your actual operational email configurations.
# For Gmail, you must use an "App Password", not your regular account password.
SMTP_SERVER = "smtp.gmail.com"
SMTP_PORT = 587
SENDER_EMAIL = "mdshadabk318@gmail.com"
SENDER_PASSWORD = "Shadab786@"
RECEIVER_EMAIL = "housing.lp@qiddiya.fmco.sa"

In [7]:
# File paths for assets (assumes logos are in the same directory)
FMCO_LOGO = "C:\\Users\\FMCO-USER.FMCO-SHARED-STG.000\Downloads\\images (1).png"
QIDDIYA_LOGO = "C:\\Users\\FMCO-USER.FMCO-SHARED-STG.000\Downloads\\FMCO-Logo-2.png"

<>:2: SyntaxWarning: invalid escape sequence '\D'
<>:3: SyntaxWarning: invalid escape sequence '\D'
<>:2: SyntaxWarning: invalid escape sequence '\D'
<>:3: SyntaxWarning: invalid escape sequence '\D'
/tmp/ipykernel_7705/331979271.py:2: SyntaxWarning: invalid escape sequence '\D'
  FMCO_LOGO = "C:\\Users\\FMCO-USER.FMCO-SHARED-STG.000\Downloads\\images (1).png"
/tmp/ipykernel_7705/331979271.py:3: SyntaxWarning: invalid escape sequence '\D'
  QIDDIYA_LOGO = "C:\\Users\\FMCO-USER.FMCO-SHARED-STG.000\Downloads\\FMCO-Logo-2.png"


In [8]:
# ==========================================
# PDF GENERATION CLASS
# ==========================================
class DepartureReportPDF(FPDF):
    def header(self):
        # Top Header Banner with Branding Logos
        if os.path.exists(FMCO_LOGO):
            self.image(FMCO_LOGO, x=10, y=10, w=35)
        if os.path.exists(QIDDIYA_LOGO):
            self.image(QIDDIYA_LOGO, x=165, y=10, w=35)

        self.ln(18)
        self.set_font("Helvetica", "B", 16)
        self.set_text_color(14, 56, 74) # Primary Deep Teal Tint
        self.cell(0, 8, "RESIDENT DEPARTURE & INSPECTION REPORT", ln=True, align="C")

        self.set_font("Helvetica", "I", 9)
        self.set_text_color(100, 100, 100)
        self.cell(0, 5, f"Generated automatically on: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}", ln=True, align="C")
        self.ln(5)

        # Decorative divider line
        self.set_draw_color(14, 56, 74)
        self.set_line_width(0.8)
        self.line(10, self.get_y(), 200, self.get_y())
        self.ln(5)

    def footer(self):
        # Position at 1.5 cm from bottom
        self.set_y(-15)
        self.set_font("Helvetica", "I", 8)
        self.set_text_color(150, 150, 150)
        self.cell(0, 10, f"Page {self.page_no()}/{{nb}} | FMCO Operational Logistics", align="C")

    def section_heading(self, label):
        self.set_font("Helvetica", "B", 12)
        self.set_text_color(255, 255, 255)
        self.set_fill_color(14, 56, 74) # Dark Teal header fill
        self.cell(0, 7, f"  {label}", ln=True, fill=True)
        self.ln(3)

In [9]:
# ==========================================
# SECURE EMAIL ENGINE
# ==========================================
def send_report_via_email(pdf_filename, occupant_name, building, room):
    msg = MIMEMultipart()
    msg['From'] = SENDER_EMAIL
    msg['To'] = RECEIVER_EMAIL
    msg['Subject'] = f"🔔 Departure Report Submitted: Bldg {building} - Room {room} ({occupant_name})"

    body = f"""
    Dear Resident Relations Team,

    A new digital departure inspection report has been successfully submitted.

    Summary Details:
    - Occupant Name: {occupant_name}
    - Building Location: {building}
    - Room Number: {room}

    The formal professional inspection PDF artifact is attached to this email transmission.
    """
    msg.attach(MIMEText(body, 'plain'))

    try:
        with open(pdf_filename, "rb") as attachment:
            part = MIMEBase('application', 'octet-stream')
            part.set_payload(attachment.read())
            encoders.encode_base64(part)
            part.add_header('Content-Disposition', f"attachment; filename= {os.path.basename(pdf_filename)}")
            msg.attach(part)

        server = smtplib.SMTP(SMTP_SERVER, SMTP_PORT)
        server.starttls()
        server.login(SENDER_EMAIL, SENDER_PASSWORD)
        server.send_message(msg)
        server.quit()
        return True
    except Exception as e:
        st.error(f"Failed to dispatch email notification: {e}")
        return False

In [11]:
with st.form("departure_form", clear_on_submit=True):

    st.subheader("🏢 Location & Occupant Core Attributes")
    col1, col2 = st.columns(2)
    with col1:
        occupant_name = st.text_input("Occupant Full Name", placeholder="John Doe")
        building_no = st.text_input("Building Number", placeholder="Bldg 404")
    with col2:
        occupant_id = st.text_input("Resident ID / Serial", placeholder="ID-8849")
        room_no = st.text_input("Room Number", placeholder="Room 201-A")

    st.markdown("---")

    st.subheader("🔍 Facility Infrastructure Checklist")
    st.markdown("*Select the structural integrity status for key room assets:*")

    # Complete categorical layout of checklist items
    structural_items = [
        "Main Door, Locks & Handles", "Wall Condition & Paint Protection",
        "Primary Flooring & Baseboards", "Windowpanes & Latches",
        "Air Conditioning (AC) Unit & Remote Function", "Lighting Fixtures & Electrical Switches",
        "Bedframe & Mattress Condition", "Wardrobe, Shelving & Cabinet Hinges",
        "Bathroom Drainage & Plumbing Assets", "Water Heater Functionality"
    ]

    checklist_results = {}
    for item in structural_items:
        status = st.radio(f"**{item}**", ["Pass (Excellent)", "Action Required (Damage/Defect)"], index=0, horizontal=True)
        checklist_results[item] = status

    st.markdown("---")

    # CORRECTED: File uploader placed cleanly inside the form layout
    st.subheader("📸 Room Condition Media Feature")
    uploaded_photo = st.file_uploader("Upload Room Condition Photo (Optional)", type=["png", "jpg", "jpeg"])

    st.markdown("---")
    st.subheader("✍️ Authorized Signatures & Remarks")
    remarks = st.text_area("Additional Field Observations / Maintenance Discrepancy Remarks", placeholder="Enter notes here...")

    col3, col4 = st.columns(2)
    with col3:
        resident_signature = st.text_input("Resident Signature (Type Name to Authorize)", placeholder="e.g., J. Doe")
    with col4:
        inspector_signature = st.text_input("Lead Inspector Signature (Type Name to Verify)", placeholder="e.g., Inspector Ahmed")

    # CORRECTED: Standard form submission button
    submit_button = st.form_submit_button("Submit Formal Report")

    if submit_button:
        if not occupant_name or not building_no or not room_no or not inspector_signature:
            st.error("❌ Critical Validation Error: Please fill in all mandatory fields (Name, Building, Room, and Inspector Signature).")
        else:
            with st.spinner("Generating professional PDF report and routing to communications hub..."):

                # Setup PDF Engine
                pdf = DepartureReportPDF()
                pdf.alias_nb_pages()
                pdf.add_page()
                pdf.set_auto_page_break(auto=True, margin=20)

                # Section 1: Metadata Table
                pdf.section_heading("1. Occupant & Operational Metadata")
                pdf.set_font("Helvetica", "", 10)
                pdf.set_text_color(50, 50, 50)

                def draw_row(left_lbl, left_val, right_lbl, right_val):
                    pdf.set_font("Helvetica", "B", 10)
                    pdf.cell(40, 7, left_lbl, border=1)
                    pdf.set_font("Helvetica", "", 10)
                    pdf.cell(55, 7, left_val, border=1)
                    pdf.set_font("Helvetica", "B", 10)
                    pdf.cell(40, 7, right_lbl, border=1)
                    pdf.set_font("Helvetica", "", 10)
                    pdf.cell(55, 7, right_val, border=1, ln=True)

                draw_row("Occupant Name:", occupant_name, "Resident ID:", occupant_id)
                draw_row("Building No:", building_no, "Room No:", room_no)
                pdf.ln(6)

                # Section 2: Inspection Status Tracker
                pdf.section_heading("2. Facility Infrastructure Checklist Evaluations")

                pdf.set_font("Helvetica", "B", 10)
                pdf.set_fill_color(230, 240, 245)
                pdf.cell(125, 7, " Evaluated Operational Asset Category", border=1, fill=True)
                pdf.cell(65, 7, " Status Verification", border=1, fill=True, ln=True)

                pdf.set_font("Helvetica", "", 9)
                for asset, eval_status in checklist_results.items():
                    pdf.cell(125, 6, f" {asset}", border=1)
                    if "Action Required" in eval_status:
                        pdf.set_text_color(180, 40, 40)
                        pdf.set_font("Helvetica", "B", 9)
                    else:
                        pdf.set_text_color(40, 110, 40)
                    pdf.cell(65, 6, f" {eval_status}", border=1, ln=True)
                    pdf.set_text_color(50, 50, 50)
                    pdf.set_font("Helvetica", "", 9)
                pdf.ln(6)

                # Section 3: Field Remarks
                if remarks.strip():
                    pdf.section_heading("3. Additional Field Inspection Remarks")
                    pdf.set_font("Helvetica", "", 10)
                    pdf.multi_cell(0, 5, remarks, border=1)
                    pdf.ln(6)

                # CORRECTED: Safely handle the uploaded file object directly
                if uploaded_photo is not None:
                    pdf.section_heading("4. Attached Room Condition Media Feature")
                    temp_img_path = f"temp_{uploaded_photo.name}"
                    with open(temp_img_path, "wb") as f:
                        f.write(uploaded_photo.getbuffer())

                    pdf.image(temp_img_path, x=15, w=110)
                    pdf.ln(4)
                    os.remove(temp_img_path)

                # Section 4/5: Signatures Block
                pdf.section_heading("Authorization Signatures Sign-Off")
                pdf.ln(2)
                pdf.set_font("Helvetica", "B", 10)
                pdf.cell(95, 5, "Resident Signature Validation:", ln=False)
                pdf.cell(95, 5, "Lead Facility Inspector Sign-Off:", ln=True)

                pdf.set_font("Helvetica", "I", 11)
                pdf.set_text_color(14, 56, 74)
                pdf.cell(95, 10, f"Signed Electronically: {resident_signature if resident_signature else 'N/A'}", ln=False)
                pdf.cell(95, 10, f"Verified Electronically: {inspector_signature}", ln=True)

                # Save and Send
                timestamp_str = datetime.now().strftime("%Y%m%d_%H%M%S")
                clean_name = occupant_name.replace(" ", "_")
                pdf_filename = f"Departure_Report_{clean_name}_{timestamp_str}.pdf"
                pdf.output(pdf_filename)

                email_success = send_report_via_email(pdf_filename, occupant_name, building_no, room_no)

                if email_success:
                    st.success(f"🎉 Success! Report for **{occupant_name}** has been processed and emailed.")
                    with open(pdf_filename, "rb") as file:
                        st.download_button(
                            label="📥 Download Generated PDF Locally",
                            data=file,
                            file_name=pdf_filename,
                            mime="application/pdf"
                        )
                else:
                    st.warning("⚠️ Form saved, but email failed to send. Check your email credentials.")

                if os.path.exists(pdf_filename):
                    os.remove(pdf_filename)

2026-05-30 18:44:27.412 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-05-30 18:44:27.413 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-05-30 18:44:27.414 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-05-30 18:44:27.415 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-05-30 18:44:27.415 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-05-30 18:44:27.416 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-05-30 18:44:27.418 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-05-30 18:44:27.418 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bar